# Two-Sample Mean Difference Testing Procedure

This notebook provides an exhaustive, auditable hypothesis-testing
procedure for independent two-sample mean differences. It runs the
full battery of tests — frequentist (parametric and non-parametric),
Bayesian, and effect sizes — and generates a report **without making
any accept/reject decision**.

## Academic Framework

This procedure follows the principles of:
- **Transparent reporting**: All tests are run and reported, avoiding
  the "garden of forking paths" problem (Gelman & Loken, 2014).
- **No automatic decisions**: The procedure reports evidence; the
  analyst interprets it.
- **Full provenance**: Data hashing and configuration logging ensure
  reproducibility (Peng, 2011).
- **Academic citations**: Every method includes its original academic
  reference.

## Step 1: Configuration

Set up the run configuration. All parameters are explicit — no hidden
defaults drive the analysis.

In [ ]:
from twosample_means.config import RunConfig, InputSpec
from twosample_means.data_io import load
from twosample_means.runner import run
from twosample_means.reporting import render_markdown, write_report

config = RunConfig(
    alpha=0.05,
    ci_level=0.95,
    hdi_mass=0.95,
    rope_width=0.01,
    mcmc_draws=2000,
    mcmc_chains=4,
    permutation_iterations=10000,
    bootstrap_iterations=10000,
    seed=42,
)
print("Configuration:")
print(f"  alpha = {config.alpha}")
print(f"  ci_level = {config.ci_level}")
print(f"  mcmc_draws = {config.mcmc_draws}")
print(f"  mcmc_chains = {config.mcmc_chains}")

## Step 2: Load Data

Load the two samples from a file (CSV/parquet) or in-memory arrays.
The loader validates the data and computes a SHA-256 provenance hash.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Load the bundled marketing A/B dataset (Kaggle:
# faviovaz/marketing-ab-testing)
#   - "test group": "ad" (treatment) vs "psa" (control)
#   - "total ads": number of ads seen by each user
data_path = Path("sample_data/marketing_AB.csv")
if not data_path.exists():
    data_path = Path("../notebooks/sample_data/marketing_AB.csv")
df = pd.read_csv(data_path)

group_a = df[df["test group"] == "ad"]["total ads"].to_numpy()
group_b = df[df["test group"] == "psa"]["total ads"].to_numpy()

print(f"Full dataset: A (ad) n={len(group_a)}, "
      f"B (psa) n={len(group_b)}")

# Subsample for notebook execution speed (MCMC on 588K rows
# would be impractical). The full dataset can be used by
# removing this subsampling step.
rng = np.random.default_rng(42)
group_a = rng.choice(group_a, size=500, replace=False)
group_b = rng.choice(group_b, size=500, replace=False)

spec = InputSpec(sample_a=group_a, sample_b=group_b)

data = load(spec)
print(f"Subsampled: A n={len(data.sample_a)}, "
      f"mean={np.mean(data.sample_a):.2f}")
print(f"Subsampled: B n={len(data.sample_b)}, "
      f"mean={np.mean(data.sample_b):.2f}")
print(f"Data hash: {data.data_hash[:16]}...")
print(f"Source: {data.source_description}")

## Step 3: Run the Full Battery

Run all tests: assumption diagnostics, parametric tests,
non-parametric tests, Bayesian tests, and effect sizes.
The runner NEVER makes an accept/reject decision.

In [ ]:
report = run(data, config)
print(f"Total results: {len(report.results)}")
categories = {}
for r in report.results:
    categories.setdefault(r.category, 0)
    categories[r.category] += 1
for cat, count in sorted(categories.items()):
    print(f"  {cat}: {count}")

## Step 4: Review Results

Display the key results from each category. The analyst interprets
these — no automatic decision is made.

In [ ]:
# Parametric tests
print("=== Parametric Tests ===")
for r in report.results:
    if r.category == "parametric" and r.statistic is not None:
        print(f"\n{r.method_name}")
        print(f"  Citation: {r.citation}")
        print(f"  Statistic: {r.statistic:.4f}")
        if r.p_value is not None:
            print(f"  p-value: {r.p_value:.6f}")
        if r.ci_lower is not None:
            print(f"  {r.ci_level*100:.0f}% CI: [{r.ci_lower:.4f}, {r.ci_upper:.4f}]")
        print(f"  Assumptions: {r.assumption_notes}")

In [ ]:
# Bayesian tests
print("=== Bayesian Tests ===")
for r in report.results:
    if r.category == "bayesian":
        print(f"\n{r.method_name}")
        print(f"  Citation: {r.citation}")
        if "bf10" in r.extra:
            print(f"  BF10: {r.extra['bf10']:.4f}")
            print(f"  BF01: {r.extra['bf01']:.4f}")
        if "rope_proportion" in r.extra:
            print(f"  Posterior mean diff: {r.statistic:.4f}")
            print(f"  HDI: [{r.ci_lower:.4f}, {r.ci_upper:.4f}]")
            print(f"  ROPE proportion: {r.extra['rope_proportion']:.4f}")
            print(f"  R-hat: {r.extra['r_hat']:.4f}")
            print(f"  ESS: {r.extra['ess']:.0f}")

In [ ]:
# Effect sizes
print("=== Effect Sizes ===")
for r in report.results:
    if r.category == "effect_size":
        print(f"\n{r.method_name}")
        print(f"  Citation: {r.citation}")
        print(f"  Estimate: {r.statistic:.4f}")
        if r.ci_lower is not None:
            print(f"  {r.ci_level*100:.0f}% CI: [{r.ci_lower:.4f}, {r.ci_upper:.4f}]")

## Step 5: Generate Report

Write the Markdown and JSON reports to disk.

In [ ]:
md_path, json_path = write_report(report, "output/")
print(f"Markdown report: {md_path}")
print(f"JSON report: {json_path}")
print(f"\n--- First 500 chars of Markdown ---")
print(render_markdown(report)[:500])

## RIGOR Assertions

The following assertions verify that the procedure meets its
auditability requirements. If any assertion fails, the procedure
is not compliant.

In [ ]:
# RIGOR assertions
assertions_passed = 0
assertions_total = 0

def assert_true(condition, name):
    global assertions_passed, assertions_total
    assertions_total += 1
    if condition:
        assertions_passed += 1
        print(f"  PASS: {name}")
    else:
        print(f"  FAIL: {name}")

# 1. Data hash is present and non-empty
assert_true(len(report.data_hash) > 0, "Data hash present")

# 2. All results have citations
all_cited = all(len(r.citation) > 0 for r in report.results)
assert_true(all_cited, "All results have citations")

# 3. No result contains an accept/reject decision
no_decisions = all(
    "reject" not in r.assumption_notes.lower()
    and "accept" not in r.assumption_notes.lower()
    for r in report.results
)
assert_true(no_decisions, "No accept/reject decisions in results")

# 4. All five categories are present
cats = {r.category for r in report.results}
assert_true("diagnostic" in cats, "Diagnostics category present")
assert_true("parametric" in cats, "Parametric category present")
assert_true("nonparametric" in cats, "Non-parametric category present")
assert_true("bayesian" in cats, "Bayesian category present")
assert_true("effect_size" in cats, "Effect size category present")

# 5. Configuration is logged
assert_true(len(report.config) > 0, "Configuration logged")

# 6. Outliers flagged but not removed
outlier_results = [
    r for r in report.results
    if "Outlier" in r.method_name
]
outliers_not_removed = all(
    "not removed" in r.assumption_notes.lower()
    for r in outlier_results
)
assert_true(outliers_not_removed, "Outliers flagged but not removed")

print(f"\n{assertions_passed}/{assertions_total} assertions passed")